# Bronze — trust metadata

`landing.trusts_raw` → `bronze.trusts`. Every column cast to STRING, nothing else.

Landing already holds this as text, so the casts are a no-op here **by design** — one
contract with no exceptions to remember beats a rule with carve-outs.

Expected: **120 rows**, matching Landing exactly.

In [0]:
CATALOG = "`index-vs-trust-pipeline`"
SOURCE = f"{CATALOG}.landing.trusts_raw"
TARGET = f"{CATALOG}.bronze.trusts"

In [0]:
src_columns = spark.table(SOURCE).columns

# Cast whatever arrived. Naming columns in advance would silently drop any the source
# adds, which is the one thing this layer must never do.
cast_list = ",\n  ".join(f"CAST(`{c}` AS STRING) AS `{c}`" for c in src_columns)
sql = f"CREATE OR REPLACE TABLE {TARGET} AS\nSELECT\n  {cast_list}\nFROM {SOURCE}"

print(f"{len(src_columns)} columns found: {src_columns}\n")
print(sql)

spark.sql(sql)
print(f"\nwrote {TARGET}")

## Verification

In [0]:
%sql
-- Bronze must reject nothing, so the two counts have to match.
SELECT
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.landing.trusts_raw) AS landing_rows,
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.bronze.trusts)      AS bronze_rows,
  (SELECT SUM(CASE WHEN ticker = '' THEN 1 ELSE 0 END)
     FROM `index-vs-trust-pipeline`.bronze.trusts)                    AS blank_tickers;

Expect **120 / 120 / 2**. The two blank tickers (Island Innovation, Witan) surviving is
the point: Bronze did not drop them.

In [0]:
%sql
-- The contract. Any non-STRING column here is a bug.
SELECT COUNT(*)                                              AS columns_total,
       SUM(CASE WHEN data_type <> 'STRING' THEN 1 ELSE 0 END) AS not_string
FROM `index-vs-trust-pipeline`.information_schema.columns
WHERE table_schema = 'bronze' AND table_name = 'trusts';

Expect **7 columns, 0 not_string**.